<a href="https://colab.research.google.com/github/phillip-jaeslee/PULSIM/blob/v_torch/PULSIM_colab_oo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# PULSIM — shaped pulse simulator

Simulate the excitation profile of shaped RF pulses and multi-pulse trains,
entirely in the browser. No installation, no licence, no local compute.

**How to use:** `Runtime` -> `Run all`, then edit the fields in the forms below and
re-run the cell you changed.

| Section | What it does |
|---|---|
| 1. Setup | Clones PULSIM and installs dependencies |
| 2. Single pulse | Pick one shape, see its excitation profile |
| 3. Pulse train | Chain several pulses, see the combined profile |
| 3b. Pulse train (widgets) | The same thing, with pulses you can add and remove |
| 4. Scripting | The same simulation written directly in code |
| 5. Shape catalogue | Every shape available, with its registry name |


## 1. Setup

Run this once per session. Takes about 30 seconds.


In [ ]:
#@title Install PULSIM { display-mode: "form" }
!git clone --quiet --branch main https://github.com/phillip-jaeslee/PULSIM
%cd PULSIM
# PULSIM installs with numpy + scipy alone. The [viz] extra adds matplotlib and
# ipywidgets, which this notebook uses for plotting.
!pip install --quiet ".[viz]"
print("PULSIM ready.")


In [ ]:
#@title Load the simulator  { display-mode: "form" }
import time

import numpy as np
import matplotlib.pyplot as plt

from PULSIM import RFShape, Pulse, PulseSequence, NumpyBackend

# Reduced gyromagnetic ratio gamma_bar = gamma / (2*pi), in kHz/mT.
# Imported rather than redefined here: the package table is the single source
# of truth, and a local copy had already drifted (19F 40.052 vs 40.078).
from PULSIM.spin_system import gyro_ratio


def make_backend(kind, Gamma):
    """NumPy is the reference implementation and the default.

    The NumPy backend batches every offset into one operation per time step, so
    it is fast enough to be interactive (see the timings at the end of this
    notebook). TorchBackend is EXPERIMENTAL: it currently moves data between
    host and device on every time step and computes in float32 where NumPy uses
    float64, so on CPU it is both slower and less accurate."""
    if kind == "torch":
        try:
            import torch  # noqa: F401
            from PULSIM import TorchBackend
            return TorchBackend(Gamma=Gamma)
        except Exception as exc:
            print(f"torch unavailable ({exc}) - falling back to numpy")
    return NumpyBackend(Gamma=Gamma)


def run_timed(seq, M, df):
    t0 = time.time()
    M = seq.run(M, df)
    print(f"  simulated in {time.time() - t0:.2f} s")
    return M


def make_M(n_offsets, init="z", M0=1.0):
    """Starting magnetization: one vector per offset, as a (3, n) array."""
    vec = {"x": [M0, 0, 0], "y": [0, M0, 0], "z": [0, 0, M0]}
    if init not in vec:
        raise ValueError(f"init must be x, y or z; got {init!r}")
    return np.tile(np.array(vec[init], dtype=float), (n_offsets, 1)).T


def plot_result(seq, M, df_khz, title=""):
    """Top: RF amplitude and phase vs time. Bottom: excitation profile vs offset."""
    fig, axs = plt.subplots(3, 1, figsize=(7, 8), height_ratios=[1, 1, 2])

    rf = np.asarray(seq.rf)
    t = np.asarray(seq.time)
    axs[0].plot(t, np.abs(rf), color="tab:blue")
    axs[0].set(xlabel="time (ms)", ylabel="|B1| (mT)", title=title or "RF amplitude")

    axs[1].plot(t, np.asarray(seq.phase), color="tab:orange")
    axs[1].set(xlabel="time (ms)", ylabel="phase (deg)")

    df_hz = df_khz * 1000.0
    axs[2].plot(df_hz, M[0], label="Mx")
    axs[2].plot(df_hz, M[1], label="My")
    axs[2].plot(df_hz, M[2], label="Mz")
    axs[2].plot(df_hz, np.hypot(M[0], M[1]), "k--", lw=1, label="|Mxy|")
    axs[2].set(xlabel="offset (Hz)", ylabel="magnetization",
               title="Excitation profile")
    axs[2].axhline(0, color="0.8", lw=0.5)
    axs[2].legend(loc="best", fontsize=8)

    fig.tight_layout()
    plt.show()
    return fig

print("Loaded.", len(RFShape.available()), "pulse shapes available.")
print("Backend: numpy (reference). Measured ~0.13 s for 400 offsets x 500 points,"
      " ~0.5 s at 1000 x 1000 -- interactive without any special measures.")


## 2. Single pulse

Pick one shape and see what it does.

`duration` is in **ms**, `bandwidth` in **kHz**, `flip` in **degrees**.
For `shape = file`, put the path to a shape file in `file_path`
(JCAMP-DX `##XYPOINTS`, JEOL `.jhl`, and Bruker-style files are all read);
otherwise leave it blank.


In [ ]:
#@title Simulate one pulse  { display-mode: "form" }
#@markdown ### Spin
nucleus   = "H"   #@param ["H", "D", "T", "13C", "15N", "19F", "31P"]
init_vect = "z"   #@param ["x", "y", "z"]

#@markdown ### Sweep
bandwidth = 6     #@param {type:"number"}
n_offsets = 1000  #@param {type:"integer"}
engine    = "numpy"  #@param ["numpy", "torch"]

#@markdown ### Pulse
shape     = "gausscasq5"  #@param ["eburp1", "eburp2", "iburp1", "iburp2", "uburp", "reburp", "gausscasg3", "gausscasg4", "gausscasq3", "gausscasq5", "hermite", "seduce1", "sneeze", "qsneeze", "esnob", "i2snob", "i3snob", "rsnob", "dsnob", "hypsec", "sinc", "cos", "hard", "file"]
duration  = 3.0   #@param {type:"number"}
points    = 1000  #@param {type:"integer"}
flip      = 90.0  #@param {type:"number"}
#@markdown Ignored for adiabatic shapes such as `hypsec`, whose amplitude comes
#@markdown from the frequency sweep and the adiabaticity factor Q instead.
axis      = "x"   #@param ["x", "y", "z"]
file_path = ""    #@param {type:"string"}

Gamma = gyro_ratio(nucleus)
df = np.linspace(-bandwidth / 2, bandwidth / 2, n_offsets)

kwargs = {"duration": duration, "points": points}
if shape == "file":
    kwargs = {"duration": duration, "path": file_path}

rf_shape = RFShape.create(shape, **kwargs)
backend = make_backend(engine, Gamma)

if rf_shape.calibration_mode == "adiabatic":
    # A flip angle is not defined for a pulse whose RF phase sweeps: the
    # Hamiltonians at different times do not commute. Amplitude comes from the
    # sweep rate and the adiabaticity factor Q instead.
    pulse = Pulse(rf_shape, axis=axis, backend=backend)
    print(f"{shape}: adiabatic. nu1_max = {pulse.nu1_max:.4f} kHz "
          f"(Q = {pulse.realized_q:.2f}); the flip angle above is ignored.")
else:
    pulse = Pulse(rf_shape, flip=np.deg2rad(flip), axis=axis, backend=backend)

seq = PulseSequence([pulse])
M = run_timed(seq, make_M(n_offsets, init_vect), df)

plot_result(seq, M, df, title=f"{shape}  {flip:g}deg  {duration:g} ms")


## 3. Pulse train

Chain several pulses. Each is applied in order to the same magnetization,
**back to back with no delay between them** — so this is a pulse *train*,
not a pulse sequence in the COSY/INEPT sense. Free precession between
pulses is not represented; see the limitations at the end.

Set `n_pulses` to how many you want, then fill in that many rows below.
Rows beyond `n_pulses` are ignored, so you can leave them alone.

In [ ]:
#@title Build a train  { display-mode: "form" }
#@markdown ### Spin and sweep
nucleus   = "H"   #@param ["H", "D", "T", "13C", "15N", "19F", "31P"]
init_vect = "z"   #@param ["x", "y", "z"]
bandwidth = 6     #@param {type:"number"}
n_offsets = 1000  #@param {type:"integer"}
engine    = "numpy"  #@param ["numpy", "torch"]
n_pulses  = 3     #@param {type:"slider", min:1, max:5, step:1}

#@markdown ---
#@markdown ### Pulse 1
shape_1 = "gausscasq5"  #@param {type:"string"}
flip_1 = 90.0    #@param {type:"number"}
dur_1 = 3.0      #@param {type:"number"}
pts_1 = 1000     #@param {type:"integer"}
axis_1 = "x"     #@param ["x", "y", "z"]
path_1 = ""      #@param {type:"string"}

#@markdown ### Pulse 2
shape_2 = "hard" #@param {type:"string"}
flip_2 = 180.0   #@param {type:"number"}
dur_2 = 0.02     #@param {type:"number"}
pts_2 = 100      #@param {type:"integer"}
axis_2 = "x"     #@param ["x", "y", "z"]
path_2 = ""      #@param {type:"string"}

#@markdown ### Pulse 3
shape_3 = "gausscasq5"  #@param {type:"string"}
flip_3 = 90.0    #@param {type:"number"}
dur_3 = 3.0      #@param {type:"number"}
pts_3 = 1000     #@param {type:"integer"}
axis_3 = "x"     #@param ["x", "y", "z"]
path_3 = ""      #@param {type:"string"}

#@markdown ### Pulse 4
shape_4 = "hard" #@param {type:"string"}
flip_4 = 90.0    #@param {type:"number"}
dur_4 = 0.02     #@param {type:"number"}
pts_4 = 100      #@param {type:"integer"}
axis_4 = "x"     #@param ["x", "y", "z"]
path_4 = ""      #@param {type:"string"}

#@markdown ### Pulse 5
shape_5 = "hard" #@param {type:"string"}
flip_5 = 90.0    #@param {type:"number"}
dur_5 = 0.02     #@param {type:"number"}
pts_5 = 100      #@param {type:"integer"}
axis_5 = "x"     #@param ["x", "y", "z"]
path_5 = ""      #@param {type:"string"}

# ---- build ----------------------------------------------------------------
Gamma = gyro_ratio(nucleus)
df = np.linspace(-bandwidth / 2, bandwidth / 2, n_offsets)
backend = make_backend(engine, Gamma)

rows = [
    (shape_1, flip_1, dur_1, pts_1, axis_1, path_1),
    (shape_2, flip_2, dur_2, pts_2, axis_2, path_2),
    (shape_3, flip_3, dur_3, pts_3, axis_3, path_3),
    (shape_4, flip_4, dur_4, pts_4, axis_4, path_4),
    (shape_5, flip_5, dur_5, pts_5, axis_5, path_5),
][:n_pulses]

seq = PulseSequence()
for i, (shp, flp, dur, pts, ax, pth) in enumerate(rows, start=1):
    shp = shp.strip().lower()
    kw = {"duration": dur, "path": pth} if shp == "file" else {"duration": dur, "points": pts}
    seq.append(Pulse(RFShape.create(shp, **kw), flip=np.deg2rad(flp), axis=ax, backend=backend))
    print(f"  {i}. {shp:<14} {flp:>6.1f} deg   {dur:>7.3f} ms   axis {ax}")

M = run_timed(seq, make_M(n_offsets, init_vect), df)
plot_result(seq, M, df, title=f"pulse train of {len(seq)} pulses")


### 3b. The same thing, with widgets

The form above is a Colab feature and caps the train at five pulses. This
version uses `ipywidgets` instead: it runs unchanged in Colab, JupyterLite and
local Jupyter, and the train can be any length — press **+ Add pulse** or **✕**.

Physics is identical: pulses back to back, no delay.

In [ ]:
#@title Build a pulse train — widget version  { display-mode: "form" }
# No Colab form fields here: only ipywidgets, so this cell behaves the same in
# Colab, JupyterLite and local Jupyter. (#@title is cosmetic — a plain comment
# outside Colab.)

import ipywidgets as widgets
from IPython.display import display, clear_output

SHAPES = sorted(RFShape.available())
if "file" not in SHAPES:
    SHAPES.append("file")

_STYLE = {"description_width": "initial"}
def _w(px):  return widgets.Layout(width=px)
def _lbl(text, px): return widgets.Label(text, layout=_w(px))

# ---- spin and sweep -------------------------------------------------------
w_nucleus = widgets.Dropdown(options=["H", "D", "T", "13C", "15N", "19F", "31P"],
                             value="H", description="nucleus:", style=_STYLE, layout=_w("140px"))
w_init    = widgets.Dropdown(options=["x", "y", "z"], value="z",
                             description="start M:", style=_STYLE, layout=_w("130px"))
w_bw      = widgets.FloatText(value=6.0, description="BW (kHz):", style=_STYLE, layout=_w("150px"))
w_noff    = widgets.IntText(value=1000, description="offsets:", style=_STYLE, layout=_w("140px"))
w_engine  = widgets.Dropdown(options=["numpy", "torch"], value="numpy",
                             description="engine:", style=_STYLE, layout=_w("140px"))

# ---- one row per pulse ----------------------------------------------------
_rows = []
pulse_box = widgets.VBox([])
out = widgets.Output()

def _refresh():
    for n, r in enumerate(_rows, start=1):
        r["idx"].value = str(n)
    pulse_box.children = tuple(r["box"] for r in _rows)

def _make_row(shape="hard", flip=90.0, dur=0.02, pts=100, axis="x", path=""):
    idx  = _lbl("", "26px")
    w_sh = widgets.Dropdown(options=SHAPES, value=shape, layout=_w("150px"))
    w_fl = widgets.FloatText(value=flip, layout=_w("80px"))
    w_du = widgets.FloatText(value=dur,  layout=_w("80px"))
    w_pt = widgets.IntText(value=pts,    layout=_w("80px"))
    w_ax = widgets.Dropdown(options=["x", "y"], value=axis, layout=_w("60px"))
    w_pa = widgets.Text(value=path, placeholder="shape file", layout=_w("150px"))
    w_rm = widgets.Button(description="✕", tooltip="remove this pulse", layout=_w("36px"))

    def _sync(change=None):
        is_file = (w_sh.value == "file")
        w_pa.disabled = not is_file      # path only matters for shape = file
        w_pt.disabled = is_file          # points come from the file itself
    w_sh.observe(_sync, names="value")
    _sync()

    def _remove(_b):
        if len(_rows) == 1:
            with out:
                clear_output(wait=True)
                print("A train needs at least one pulse.")
            return
        _rows.remove(row)
        _refresh()
    w_rm.on_click(_remove)

    row = {"idx": idx, "shape": w_sh, "flip": w_fl, "dur": w_du, "pts": w_pt,
           "axis": w_ax, "path": w_pa,
           "box": widgets.HBox([idx, w_sh, w_fl, w_du, w_pt, w_ax, w_pa, w_rm])}
    return row

def _add(_b=None):
    # a new row copies the last one, so repeated pulses are two clicks
    if _rows:
        p = _rows[-1]
        _rows.append(_make_row(p["shape"].value, p["flip"].value, p["dur"].value,
                               p["pts"].value, p["axis"].value, p["path"].value))
    else:
        _rows.append(_make_row())
    _refresh()

# ---- run ------------------------------------------------------------------
def _run(_b):
    with out:
        clear_output(wait=True)
        try:
            print("Run button has been clicked. Now running... Please wait...")
            Gamma   = gyro_ratio(w_nucleus.value)
            backend = make_backend(w_engine.value, Gamma)
            n       = int(w_noff.value)
            df      = np.linspace(-w_bw.value / 2, w_bw.value / 2, n)

            seq = PulseSequence()
            for i, r in enumerate(_rows, start=1):
                name = r["shape"].value
                if name == "file":
                    kw = {"duration": r["dur"].value, "path": r["path"].value.strip()}
                else:
                    kw = {"duration": r["dur"].value, "points": int(r["pts"].value)}
                shp = RFShape.create(name, **kw)

                if shp.calibration_mode == "adiabatic":
                    # No flip angle for a phase-swept pulse: amplitude comes from
                    # the sweep rate and Q instead.
                    p = Pulse(shp, axis=r["axis"].value, backend=backend)
                    print(f"  {i}. {name:<14} adiabatic  {r['dur'].value:>7.3f} ms  "
                          f"axis {r['axis'].value}   (nu1_max {p.nu1_max:.4f} kHz, "
                          f"Q {p.realized_q:.2f}; flip ignored)")
                else:
                    p = Pulse(shp, flip=np.deg2rad(r["flip"].value),
                              axis=r["axis"].value, backend=backend)
                    print(f"  {i}. {name:<14} {r['flip'].value:>6.1f} deg  "
                          f"{r['dur'].value:>7.3f} ms  axis {r['axis'].value}")
                seq.append(p)

            M = run_timed(seq, make_M(n, w_init.value), df)
            plot_result(seq, M, df, title=f"pulse train of {len(seq)} pulses")
        except Exception as exc:
            print(f"{type(exc).__name__}: {exc}")

b_add = widgets.Button(description="+ Add pulse", layout=_w("120px"))
b_run = widgets.Button(description="Run", button_style="primary", layout=_w("120px"))
b_add.on_click(_add)
b_run.on_click(_run)

# ---- default train: the same one the form cell starts with ----------------
for _args in [("gausscasq5", 90.0, 3.0, 1000, "x", ""),
              ("hard",      180.0, 0.02, 100, "x", ""),
              ("gausscasq5", 90.0, 3.0, 1000, "x", "")]:
    _rows.append(_make_row(*_args))
_refresh()

header = widgets.HBox([_lbl("#", "26px"), _lbl("shape", "150px"), _lbl("flip °", "80px"),
                       _lbl("dur ms", "80px"), _lbl("points", "80px"),
                       _lbl("axis", "60px"), _lbl("file path", "150px")])

display(widgets.VBox([
    widgets.HTML("<b>Spin and sweep</b>"),
    widgets.HBox([w_nucleus, w_init, w_bw, w_noff, w_engine]),
    widgets.HTML("<b>Pulses</b> — applied in order, back to back, no delay"),
    header, pulse_box,
    widgets.HBox([b_add, b_run]),
    out,
]))

## 4. The same thing, in code

The two form above is convenience. Underneath, a whole train is this:

```python
seq = PulseSequence([
    Pulse(RFShape.create("gausscasq5", duration=3.0,  points=1000), np.pi/2),
    Pulse(RFShape.create("hard",       duration=0.02, points=100),  np.pi),
    Pulse(RFShape.create("gausscasq5", duration=3.0,  points=1000), np.pi/2),
])
M = seq.run(make_M(1000, "z"), df)
```

Which makes comparisons easy — here is every Gaussian-cascade member at once.


In [ ]:
#@title Compare several shapes
bandwidth = 6
n_offsets = 600
df = np.linspace(-bandwidth / 2, bandwidth / 2, n_offsets)
backend = make_backend("numpy", gyro_ratio("H"))

to_compare = ["gausscasg3", "gausscasg4", "gausscasq3", "gausscasq5"]

fig, ax = plt.subplots(figsize=(7, 4))
for name in to_compare:
    seq = PulseSequence([
        Pulse(RFShape.create(name, duration=3.0, points=1000),
              flip=np.deg2rad(90), backend=backend)
    ])
    M = seq.run(make_M(n_offsets, "z"), df)
    ax.plot(df * 1000, np.hypot(M[0], M[1]), label=name)

ax.set(xlabel="offset (Hz)", ylabel="|Mxy|",
       title="90 deg, 3 ms — Gaussian cascade family")
ax.legend()
fig.tight_layout()
plt.show()


## 5. Shape catalogue

Every registered shape name, as accepted by `RFShape.create(...)`.


In [ ]:
#@title List every available shape { display-mode: "form" }
names = sorted(RFShape.available())
print(f"{len(names)} shapes registered\n")
for i in range(0, len(names), 4):
    print("  " + "".join(f"{n:<24}" for n in names[i:i + 4]))


---

### Known limitations of this version

- **No delay element.** A `PulseSequence` chains pulses back to back, which makes
  it a pulse *train*; a gap between pulses (free precession) cannot yet be
  represented. Real pulse sequences with inter-pulse delays are therefore out of
  scope for this version.
- **`axis` accepts `"x"`, `"y"`, or a phase in radians.** `"z"` raises: it was never
  used and has no independent ground truth to check against.
- **No relaxation during pulses.** T1 and T2 are ignored inside a pulse.
- **Adiabatic shapes.** Only `hypsec` has a derived sweep-rate calibration. The other
  adiabatic families (WURST, tanh/tan, the constant-adiabaticity shapes) raise
  `NotImplementedError` rather than guessing: each needs its own expression. Supply
  `nu1_max` explicitly to simulate them. Half-passage sweeps are not implemented.
- **Speed.** The NumPy backend batches every offset into one operation per time step.
  Measured: ~0.13 s for 400 offsets x 500 points, ~0.5 s at 1000 x 1000.
- **Torch is experimental.** `TorchBackend` moves data between host and device on
  *every* time step and computes in float32 where NumPy uses float64, so on CPU it is
  both slower and less accurate than the default. No GPU benefit has been measured.

### Citing

If PULSIM is useful in your work, please cite the repository:
<https://github.com/phillip-jaeslee/PULSIM>
